In [1]:
import sys, os
import tensorflow as tf
import numpy as np
from core.rollout import *
from core.agents import *
sys.path.insert(0, os.path.abspath("../.."))   # sandboxes/test2 -> project root
from game_environments.ToyGame import ToyGame
from beliefs import categorical_eirmappo

In [2]:
batch = 3
game = ToyGame(batch, ep_len=5)
agents = build_agents(game, categorical_eirmappo, hidden=128)
belief_step = take_belief_step(agents, categorical_eirmappo)

In [39]:
def run_episodes(game, agents, belief_step, hidden=128):
    """Run B episodes in parallel. Returns dict of (B, T, ...) arrays."""
    game.reset()
    theta = generate_types(game)           # choose corrupted agent
    prev_actions = np.zeros((game.B, game.N), dtype=int)
    h = tf.zeros((game.B * game.N, hidden))
    S_state, S_belief, S_actions, S_reward, S_feat = [], [], [], [], []

    for t in range(game.ep_len):
        obs_all = game._obs()
        state_oh = obs_all[:, 0, :]     # (B, n_states)  all agents see the same state
        # Compute belief based on observations
        f = belief_feats(state_oh, prev_actions, game)
        policy_input, h, b_probs = belief_step(tf.constant(f), h)
        policy_input_np = policy_input.numpy()
        # print("Policy Input (belief):\n", policy_input_np)

        # cooperative actions: policy conditions on the belief representation
        # (raw hidden state for factored belief, softmax probs for categorical belief)
        state_rep = np.repeat(state_oh, game.N, axis=0)   # (B*N, n_states)
        c_logits  = agents.policy([state_rep, policy_input_np]).numpy()      # (B*N, A)
        a_coop    = choose_action(c_logits).reshape(game.B, game.N)

        # --- adversary actions: one batched call, one batched sample ---
        a_logits = agents.adversary(obs_all.reshape(game.B*game.N, game.n_states)).numpy()  # (B*N, A)
        a_adv    = choose_action(a_logits).reshape(game.B, game.N)                        # (B, N)

        # --- apply corruption ---
        actions = np.where(theta == 1, a_adv, a_coop)
        _, r, done = game.step(actions)
        
        S_state.append(state_oh); S_belief.append(b_probs.numpy().reshape(game.B, game.N, -1))
        S_actions.append(actions.copy()); S_reward.append(r); S_feat.append(f)
        prev_actions = actions
        if done: break

    stack = lambda L: np.stack(L, axis=1)
    return dict(
        state=stack(S_state), belief=stack(S_belief),
        actions=stack(S_actions), reward=stack(S_reward), feat=stack(S_feat),
        theta=np.repeat(theta[:, None, :], len(S_reward), axis=1),
    )

eps = run_episodes(game, agents, belief_step)
cols = eps.keys()
print(cols)
print('Episode reward mean: ',eps['reward'].mean(axis=1))
has_adv = eps['theta'][:, 0, :].sum(axis=1) > 0
print('Which run had an adversary: ',has_adv)
eps['reward'].mean(axis=1)[has_adv].mean()  if has_adv.any()   else float('nan')
# print(eps['theta'])
# print(eps)

dict_keys(['state', 'belief', 'actions', 'reward', 'feat', 'theta'])
Episode reward mean:  [0.4 0.8 0.4]
Which run had an adversary:  [ True False False]


0.4

In [40]:
print(has_adv)
print(eps['reward'].mean(axis=1))
print(eps['reward'].mean(axis=1)[has_adv])
print(eps['reward'].mean(axis=1)[has_adv].mean())

[ True False False]
[0.4 0.8 0.4]
[0.4]
0.4
